In [ ]:
# STEP 1: Install the Agents SDK (run this cell first)
# Remove any old version of the SDK
!pip uninstall -y openai-agents

# Install the exact version used in this course
%pip install --no-cache-dir openai-agents==0.2.2

# python-dotenv reads the .env file; LangChain is used in later modules
%pip install python-dotenv langchain-openai==0.2.1

# Pydantic validates and organises data (used later for structured outputs)
%pip install --upgrade pydantic

In [ ]:
# STEP 2: Connect to Groq (free key from console.groq.com/keys)
import os
from openai import OpenAI, AsyncOpenAI
from IPython.display import display, Markdown
from google.colab import userdata                 # Colab's secret store
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

groq_api_key = userdata.get("GROQ_API_KEY")       # secret name in Colab: GROQ_API_KEY
GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq's OpenAI-compatible address

# Same OpenAI library, different address: requests now go to Groq, not OpenAI
groq_client = AsyncOpenAI(api_key = groq_api_key, base_url = GROQ_BASE_URL)

# Traces are uploaded to OpenAI's dashboard and need an OpenAI key, so switch them off
set_tracing_disabled(True)

print("Groq client successfully configured.")
print(groq_api_key[:5])                           # should print: gsk_x

In [ ]:
# STEP 3: Choose the models and see which ones your key can use
GROQ_MODEL     = "openai/gpt-oss-20b"     # small and very fast  (replaces gpt-4.1-mini)
GROQ_BIG_MODEL = "openai/gpt-oss-120b"    # bigger and smarter   (replaces the larger GPT models)

def groq(model_name):
    """Wraps a Groq model name so the Agents SDK sends it to Groq exactly as written."""
    return OpenAIChatCompletionsModel(model = model_name, openai_client = groq_client)

# Model names change over time. If you get "404 model not found", pick a name from this list.
sync_client = OpenAI(api_key = groq_api_key, base_url = GROQ_BASE_URL)
for m in sync_client.models.list().data:
    print(m.id)

In [ ]:
from agents import Agent

market_researcher_instructions = """
Context:
You are a market research assistant helping analyze companies, industries, and competitors.

Instructions:
When given a question, provide a short factual answer based on your knowledge.

Output:
Start with a verdict prefix: either "✅ FACT:" or "❌ UNKNOWN:"
Follow with a concise one-sentence explanation.
"""

market_researcher_agent = Agent(name = "Market Researcher",
                                instructions = market_researcher_instructions,
                                model = groq(GROQ_MODEL))           # Groq instead of "gpt-4.1-mini"

In [ ]:
def print_markdown(text):
    """Displays text as Markdown in Jupyter."""
    display(Markdown(text))

def show_usage(result):
    """Our replacement for OpenAI traces: how many requests and tokens did the last run use?"""
    u = result.context_wrapper.usage
    print(f"📊 Requests: {u.requests} | Input tokens: {u.input_tokens} | Output tokens: {u.output_tokens}")

In [ ]:
from agents import Runner, SQLiteSession

q1 = "What is the market share of Tesla in the US EV market?"
resp1 = await Runner.run(starting_agent = market_researcher_agent, input = q1)
print_markdown(f"🤖 Agent:\n{resp1.final_output}")

q2 = "How does that compare to last year?"
resp2 = await Runner.run(starting_agent = market_researcher_agent, input = q2)
print_markdown(f"🤖 Agent:\n{resp2.final_output}")
show_usage(resp2)

In [ ]:
# Create a session instance (the agent's conversation diary)
session = SQLiteSession("conversation")

# First interaction
user_input1 = "What is the market share of Tesla in the US EV market?"
response1 = await Runner.run(starting_agent = market_researcher_agent,
                             input = user_input1,
                             session = session)        # <- NEW
print_markdown(f"🤖 Agent:\n{response1.final_output}")

# Second interaction - same session, so the agent sees the history
user_input2 = "How does that compare to last year?"
response2 = await Runner.run(starting_agent = market_researcher_agent,
                             input = user_input2,
                             session = session)
print_markdown(f"🤖 Agent:\n{response2.final_output}")
show_usage(response2)                                   # more input tokens: the history is sent too

In [ ]:
# Open the diary: what is stored in the session?
items = await session.get_items()
print(f"The session holds {len(items)} items:\n")
for item in items:
    role = item.get("role", item.get("type", "?"))
    content = item.get("content", "")
    if isinstance(content, list):                       # assistant messages store a list of parts
        content = " ".join(part.get("text", "") for part in content if isinstance(part, dict))
    print(f"{role:>9}: {str(content)[:100]}")

In [ ]:
# Wipe the diary, then ask the follow-up again
await session.clear_session()

response3 = await Runner.run(starting_agent = market_researcher_agent,
                             input = "How does that compare to last year?",
                             session = session)
print_markdown(f"🤖 Agent:\n{response3.final_output}")